# Setup

In [ ]:
import sys
sys.path.append('../../')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import pandas as pd
from pandas import DataFrame
from processor.llm.interface.model_interface import ModelInterface
from processor.llm.interface.model_factory import get_model
from processor.utils import format_schema_with_samples
from processor.types.message import Message

In [ ]:
ckp = '../llm/weight/qwen25-7b'
interface = get_model(ckp)
model: ModelInterface = interface(ckp)
model.load_model()
model.load_tokenizer()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
base_table_producer_prompts = {
    "tables_selector": """You are an experienced data scientist. You are given:
- A table, represented by its schema, a description of what it contains, and some sample rows. The pipe character (`|`) is used as the separator for both columns and row values.
- A target schema that needs to be constructed using one or more of the available tables.

Your task is to determine whether this table is **relevant** for constructing the target schema — either fully or partially. A table is considered relevant if it provides **any** useful information toward fulfilling the target schema, such as:
- Matching any of the target columns exactly,
- Providing a column that can be transformed into a target column,
- Contributing auxiliary information (e.g., geographic clues from `city` or `address` that help construct `Is in Bay Area`).

Err on the side of inclusion: if you think even **one** column might help, mark the table as **relevant**.

End your reasoning with the following exact format, to ease parsing:

Relevant: yes/no
""",
  "row_extender_step_1": """You are an experienced data scientist. You are given:
- A list of tables, each with its schema, a short description, and a few sample rows.
- The pipe character (`|`) is used to separate both column names and values.

Your task is to **analyze and describe** what each table represents, and then identify **which tables describe the same kind of real-world entity or object** (such as people, products, companies, events, etc.).

Only group tables that:
- Refer to the same kind of entity
- Can be combined via **row extension** (i.e., vertical stacking)
- Even if the columns are not exactly the same, their rows should be logically stackable (e.g., two tables of products with different attributes)

Do **not** group tables that refer to different concepts/entities, even if they share similar-looking columns.

Finish with a list of compatible groups like:
Row extension groups: Group 1: Table_0, Table_2 Group 2: Table_3, Table_4 ... (or none if no combinations are found)""",
  "row_extender_step_2": """You are an experienced data scientist. You have already analyzed the tables and identified which ones can be combined via row extension (i.e., vertically stacked) because they refer to the same kind of real-world entity.

You are given:
- A list of tables (description + schemas + samples)
- Your own prior reasoning and a list of row-extension groups (e.g., Group 1: Table_0, Table_2)

Your job is to create a JSON plan that shows how each group can be merged via row extension.

Instructions:
- For each group, create a **unified schema** by merging **semantically equivalent** columns (e.g., "Customer_Rating" and "RATING" should both become "Rating")
- Use **simple, general, and meaningful** names for the unified columns (e.g., "Phone", "Address", "Rating", "Reviews")
- For each table, create a mapping from its original column names to the unified schema
- It's okay if some original columns do not exist in the unified schema — just leave them unmapped
- Do not include duplicate columns in the unified schema — each concept should appear only once

Output directly the following format without extra texts or explanations:

Format if row extension groups exist:
```json
[
  {
    "Tables": ["Table_0", "Table_2"],
    "Unified Schema": ["Column1", "Column2", ...],
    "Mappings": {
      "Table_0": {"OrigColA": "Column1", "OrigColB": "Column2", ...},
      "Table_2": {"ColX": "Column1", "ColY": "Column2", ...}
    }
  }
]```

Format if row extension groups are empty/none:
```json
[]```""",
  "join_planner": """You are a highly skilled data engineer. You are given:
- A list of tables (with descriptions, schemas, and sample rows)
- The goal is to **join all tables** together into a final unified table by **step-wise horizontal merging**.

Assumptions:
- All tables should be joinable via appropriate key columns, either directly or through intermediate tables.
- You can choose any join order as long as all tables are included by the end.
- You should identify the most appropriate **key columns** for joining each pair of tables based on semantics or value similarity.
- The operations will be carried out using either SQL or semantic joins.

Your task:
- Construct a step-by-step join plan as a **list of operations**, where each operation joins two tables (or previous join results).
- Each step should specify:
  - The two input tables, one of which may be a join result from the prior step.
  - The columns being used for the join
  - The resulting table name for that step (e.g., "Join_1", "Join_2", etc.)

Output your answer directly as a JSON object with the following format without any extra explanations or formatting:

```json
[
  {
    "Join Result": "Join_1",
    "Left Table": "Table_A",
    "Right Table": "Table_B",
    "Left Join Key": "Column_X",
    "Right Join Key": "Column_Y"
  },
  {
    "Join Result": "Join_2",
    "Left Table": "Join_1",
    "Right Table": "Table_C",
    "Left Join Key": "UserID",
    "Right Join Key": "Customer_ID"
  }
]```"""
}

# Tables Selection

In [ ]:
def __select_relevant_tables(available_tables: list[DataFrame], tables_descs: list[str], target_schema: list[str]):
    relevant_tables: list[DataFrame] = []
    for table_idx, table in enumerate(available_tables):
        msg: list[Message] = [
            {'role': 'system', 'content': base_table_producer_prompts['tables_selector']},
            {'role': 'user', 'content': f"- Table: {format_schema_with_samples(table)}\n\n- Target schema: {target_schema}\n\n- Description: {tables_descs[table_idx]}" },
        ]
        table_relevancy_output = model.chat(msg)
        print(f"=> table_relevancy_output: {table_relevancy_output}")
        table_relevance = table_relevancy_output.split('Relevant: ')[-1].lower().strip()
        if table_relevance.startswith('yes'):
            print(f"==> Yes, this table is relevant!")
            relevant_tables.append(True)
        else:
            relevant_tables.append(False)
    return relevant_tables

## Research Synthetic

In [ ]:
prefix = "../../../data_src/synthetic_research_sem_joins"
suffix = "_enhanced.csv"
institutions = pd.read_csv(f"{prefix}/institutions{suffix}")
research_projects = pd.read_csv(f"{prefix}/research_projects{suffix}")
scientists = pd.read_csv(f"{prefix}/scientists{suffix}")

available_tables = [institutions, research_projects, scientists]
tables_descs = [
    'This table likely represents a list of universities and their corresponding countries.',
    'This table likely represents a list of scientific research projects, including the project ID, title of the project, and the lead scientist(s) responsible.',
    'The table likely represents a list of researchers or experts, including their names, fields of expertise, and affiliations with educational institutions or research organizations.',
]
question = 'List all projects led by scientists working in water purification or bioenergy, and show the full institution names and their countries.'
target_schema = ['project_title', 'scientist_name', 'scientist_field', 'institution_name', 'institution_country']
relevant_tables = __select_relevant_tables(available_tables, tables_descs, target_schema)
relevant_tables

=> table_relevancy_output: This table contains information about universities and their countries. It can contribute to the target schema by providing the institution's name and country, which match two of the target columns exactly.

Relevant: yes
==> Yes, this table is relevant!
=> table_relevancy_output: Let's analyze the table and see how it aligns with the target schema:

1. **Project_Title**: This column matches directly with the target column 'project_title'.
2. **Lead_Scientist(s)**: This column contains names of scientists who are leading the projects. We can extract the name part of this field to match the target column 'scientist_name'.
3. **Other Columns**: There are no other columns in the provided table that directly correspond to the target columns 'scientist_field', 'institution_name', or 'institution_country'.

Given that we can use the 'Project_Title' and 'Lead_Scientist(s)' columns to partially fulfill the target schema, we should consider this table relevant.

Relev

[True, True, True]

## Zomato-Yelp

In [ ]:
zomato = pd.read_csv("../../../data_src/zomato.csv")
zomato.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "Customer_Rating",
    "TELEPHONE_NUMBER",
    "NUMBER_OF_REVIEWS",
    "Full Address",
]
yelp = pd.read_csv("../../../data_src/yelp.csv")
yelp.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "RESTAURANT_RATING",
    "PHONE_NUMBER_FORMATTED",
    "REVIEWS",
    "LOCATION",
]

available_tables = [zomato, yelp]
tables_descs = [
    'This table likely represents restaurant listings including ratings, contact information, and addresses.',
    'This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.',
]
target_schema = ['Restaurant ID', 'Restaurant Name', 'Ratings', 'Is in Bay Area']
relevant_tables = __select_relevant_tables(available_tables, tables_descs, target_schema)
relevant_tables  # Yes, Yes!

=> table_relevancy_output: This table contains columns that match or can be transformed to match the target schema:

1. `RESTAURANT_ID` can be mapped directly to `Restaurant ID`.
2. `RESTAURANT_NAME` can be mapped directly to `Restaurant Name`.
3. `Customer_Rating` can be mapped to `Ratings`.
4. While `Full Address` does not directly map to `Is in Bay Area`, it provides the necessary geographic information to determine if a restaurant is in the Bay Area.

Given these mappings, the table provides useful information towards constructing the target schema.

Relevant: yes
==> Yes, this table is relevant!
=> table_relevancy_output: The table contains columns that match or can be transformed to match the target schema:

- `RESTAURANT_ID` matches 'Restaurant ID'
- `RESTAURANT_NAME` matches 'Restaurant Name'
- `RESTAURANT_RATING` can be transformed to 'Ratings' (by removing the decimal part)
- The `LOCATION` column can provide auxiliary information to determine if a restaurant is in the Bay Ar

[True, True]

# Row Extension

In [ ]:
def __union_tables(available_tables: list[DataFrame], tables_descs: list[str]):
    available_tables_formatted = ""
    for table_idx, table in enumerate(available_tables):
        available_tables_formatted += f"- Table {table_idx} ({tables_descs[table_idx]}):\n```{format_schema_with_samples(table)}```\n\n"
    available_tables_formatted = available_tables_formatted.strip()
    print(f"=> available_tables_formatted: {available_tables_formatted}")

    msg: list[Message] = [
        {'role': 'system', 'content': base_table_producer_prompts['row_extender_step_1']},
        {'role': 'user', 'content': available_tables_formatted},
    ]
    reasoning = model.chat(msg)
    print(f"=> reasoning: {reasoning}")
    msg: list[Message] = [
        {'role': 'system', 'content': base_table_producer_prompts['row_extender_step_2']},
        {'role': 'user', 'content': f'- Tables: {available_tables_formatted}\n\n- Reasoning: {reasoning}'},
    ]
    operations = model.chat(msg)
    print(f"=> operations: {operations}")
    return operations

## Research-Synthetic

In [ ]:
prefix = "../../../data_src/synthetic_research_sem_joins"
suffix = "_enhanced.csv"
institutions = pd.read_csv(f"{prefix}/institutions{suffix}")
research_projects = pd.read_csv(f"{prefix}/research_projects{suffix}")
scientists = pd.read_csv(f"{prefix}/scientists{suffix}")

available_tables = [institutions, research_projects, scientists]
tables_descs = [
    'This table likely represents a list of universities and their corresponding countries.',
    'This table likely represents a list of scientific research projects, including the project ID, title of the project, and the lead scientist(s) responsible.',
    'The table likely represents a list of researchers or experts, including their names, fields of expertise, and affiliations with educational institutions or research organizations.',
]
extend_operations = __union_tables(available_tables, tables_descs)
extend_operations  # Shouldn't be extended; no equivalent tables

=> available_tables_formatted: - Table 0 (This table likely represents a list of universities and their corresponding countries.):
```col: University_Name | University_Country
sample row 1: University of Illinois Urbana-Champaign | USA
sample row 2: Stanford University | USA
sample row 3: University of California, Berkeley | USA```

- Table 1 (This table likely represents a list of scientific research projects, including the project ID, title of the project, and the lead scientist(s) responsible.):
```col: Project_ID | Project_Title | Lead_Scientist(s)
sample row 1: R009 | Soil regeneration systems | GeoBotanist
sample row 2: R002 | Quantum encryption | Prof. Alicia B
sample row 3: R006 | Renewable water filters | Ana N.```

- Table 2 (The table likely represents a list of researchers or experts, including their names, fields of expertise, and affiliations with educational institutions or research organizations.):
```col: ResearcherName | Expertise | Institution Affiliation
sample row 

=> reasoning: Based on the provided descriptions and sample rows, we can analyze which tables represent the same kind of real-world entities and can be combined via row extension.

### Analysis:
- **Table 0**: Represents universities and their countries.
- **Table 1**: Represents scientific research projects and their lead scientists.
- **Table 2**: Represents researchers and their affiliations with educational institutions or research organizations.

### Compatibility Check:
- **Table 0** deals with universities and does not have any overlap with the other tables in terms of entities.
- **Table 1** and **Table 2** both deal with researchers but in different contexts (projects vs. affiliations). While they share the `ResearcherName` field, the context and additional information in each table make them incompatible for row extension.

### Conclusion:
None of the tables can be combined via row extension because they represent different kinds of entities and do not share enough commonalit

'[]'

## Yelp-Zomato

In [ ]:
zomato = pd.read_csv("../../../data_src/zomato.csv")
zomato.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "Customer_Rating",
    "TELEPHONE_NUMBER",
    "NUMBER_OF_REVIEWS",
    "Full Address",
]
yelp = pd.read_csv("../../../data_src/yelp.csv")
yelp.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "RESTAURANT_RATING",
    "PHONE_NUMBER_FORMATTED",
    "REVIEWS",
    "LOCATION",
]

available_tables = [zomato, yelp]
tables_descs = [
    "This table likely represents restaurant listings including ratings, contact information, and addresses.",
    "This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.",
]
extend_operations = __union_tables(available_tables, tables_descs)
extend_operations  # Yup, Extend both

=> available_tables_formatted: - Table 0 (This table likely represents restaurant listings including ratings, contact information, and addresses.):
```col: RESTAURANT_ID | RESTAURANT_NAME | Customer_Rating | TELEPHONE_NUMBER | NUMBER_OF_REVIEWS | Full Address
sample row 1: 1450000000291 | Big & Little's  | 3.8 | (773) 857-6677 | 36 | 1034 W. Belmont Avenue, Chicago, IL
sample row 2: 1450000002328 | Salonica  | 3.8 | (773) 752-3899 | 130 | 1440 E. 57th Street, Chicago, IL
sample row 3: 1450000001462 | La Brioche  | 3.6 | (608) 233-3388 | 257 | 2862 University Ave, Madison, WI```

- Table 1 (This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.):
```col: RESTAURANT_ID | RESTAURANT_NAME | RESTAURANT_RATING | PHONE_NUMBER_FORMATTED | REVIEWS | LOCATION
sample row 1: 1445980005373 | The Village Idiot  | 3.5 | (323) 655-3331 | 958 | 7383 Melrose Ave, Los Angeles, CA 90046
sample row 2: 1445980005301 

=> reasoning: Row extension groups: Group 1: Table_0, Table_1

Both Table_0 and Table_1 represent information about restaurants, including details such as the restaurant ID, name, rating, contact information, number of reviews, and location. Although the exact column names differ slightly, the information provided in each table can be logically stacked together by matching the `RESTAURANT_ID` or `RESTAURANT_NAME`. Therefore, these two tables can be combined via row extension.
=> operations: ```json
[
  {
    "Tables": ["Table_0", "Table_1"],
    "Unified Schema": ["Restaurant_ID", "Restaurant_Name", "Rating", "Phone_Number", "Number_of_Reviews", "Location"],
    "Mappings": {
      "Table_0": {"RESTAURANT_ID": "Restaurant_ID", "RESTAURANT_NAME": "Restaurant_Name", "Customer_Rating": "Rating", "TELEPHONE_NUMBER": "Phone_Number", "NUMBER_OF_REVIEWS": "Number_of_Reviews", "Full Address": "Location"},
      "Table_1": {"RESTAURANT_ID": "Restaurant_ID", "RESTAURANT_NAME": "Restaurant_Name",

'```json\n[\n  {\n    "Tables": ["Table_0", "Table_1"],\n    "Unified Schema": ["Restaurant_ID", "Restaurant_Name", "Rating", "Phone_Number", "Number_of_Reviews", "Location"],\n    "Mappings": {\n      "Table_0": {"RESTAURANT_ID": "Restaurant_ID", "RESTAURANT_NAME": "Restaurant_Name", "Customer_Rating": "Rating", "TELEPHONE_NUMBER": "Phone_Number", "NUMBER_OF_REVIEWS": "Number_of_Reviews", "Full Address": "Location"},\n      "Table_1": {"RESTAURANT_ID": "Restaurant_ID", "RESTAURANT_NAME": "Restaurant_Name", "RESTAURANT_RATING": "Rating", "PHONE_NUMBER_FORMATTED": "Phone_Number", "REVIEWS": "Number_of_Reviews", "LOCATION": "Location"}\n    }\n  }\n]\n```'

## Implement Operations

In [53]:
# Parse the operation
import json
operation_json = '```json\n[\n  {\n    "Tables": ["Table_0", "Table_1"],\n    "Unified Schema": ["Restaurant_ID", "Restaurant_Name", "Rating", "Phone_Number", "Number_of_Reviews", "Location"],\n    "Mappings": {\n      "Table_0": {"RESTAURANT_ID": "Restaurant_ID", "RESTAURANT_NAME": "Restaurant_Name", "Customer_Rating": "Rating", "TELEPHONE_NUMBER": "Phone_Number", "NUMBER_OF_REVIEWS": "Number_of_Reviews", "Full Address": "Location"},\n      "Table_1": {"RESTAURANT_ID": "Restaurant_ID", "RESTAURANT_NAME": "Restaurant_Name", "RESTAURANT_RATING": "Rating", "PHONE_NUMBER_FORMATTED": "Phone_Number", "REVIEWS": "Number_of_Reviews", "LOCATION": "Location"}\n    }\n  }\n]\n```'
if operation_json.startswith('```'):
    operation_json = operation_json[3:]
if operation_json.endswith('```'):
    operation_json = operation_json[:-3]
if operation_json.startswith('json'):
    operation_json = operation_json[4:]
operations = json.loads(operation_json)

In [55]:
extended_dfs = []
tables = {
    "Table_0": zomato,
    "Table_1": yelp,
}
last_idx = 1
for operation in operations:
    normalized_tables = []
    for table_name in operation["Tables"]:
        df = tables[table_name]
        mapping = operation["Mappings"][table_name]

        renamed_df = df.rename(columns=mapping)
        schema = operation["Unified Schema"]
        for col in schema:
            if col not in renamed_df.columns:
                renamed_df[col] = None

        renamed_df = renamed_df[schema]
        normalized_tables.append(renamed_df)
    extended_df = pd.concat(normalized_tables, ignore_index=True)
    extended_dfs.append(extended_df)
for extended_df in extended_dfs:
    tables[f'Table_{last_idx+1}'] = extended_df
    last_idx += 1
tables['Table_2'].to_csv('Yelp-Zomato.csv', index=False)

# Semantic Join Operations

## Planning

In [ ]:
def __sem_join_tables(available_tables: list[DataFrame], tables_descs: list[str]):
    available_tables_formatted = ""
    for table_idx, table in enumerate(available_tables):
        available_tables_formatted += f"- Table {table_idx} ({tables_descs[table_idx]}):\n```{format_schema_with_samples(table)}```\n\n"
    available_tables_formatted = available_tables_formatted.strip()
    print(f"=> available_tables_formatted: {available_tables_formatted}")

    msg: list[Message] = [
        {'role': 'system', 'content': base_table_producer_prompts['join_planner']},
        {'role': 'user', 'content': available_tables_formatted},
    ]
    plan = model.chat(msg)
    print(f"=> plan: {plan}")
    return plan

In [63]:
prefix = "../../../data_src/synthetic_research_sem_joins"
suffix = "_enhanced.csv"
institutions = pd.read_csv(f"{prefix}/institutions{suffix}")
research_projects = pd.read_csv(f"{prefix}/research_projects{suffix}")
scientists = pd.read_csv(f"{prefix}/scientists{suffix}")

available_tables = [institutions, research_projects, scientists]
tables_descs = [
    'This table likely represents a list of universities and their corresponding countries.',
    'This table likely represents a list of scientific research projects, including the project ID, title of the project, and the lead scientist(s) responsible.',
    'The table likely represents a list of researchers or experts, including their names, fields of expertise, and affiliations with educational institutions or research organizations.',
]
extend_operations = __sem_join_tables(available_tables, tables_descs)
extend_operations

=> available_tables_formatted: - Table 0 (This table likely represents a list of universities and their corresponding countries.):
```col: University_Name | University_Country
sample row 1: University of Illinois Urbana-Champaign | USA
sample row 2: Stanford University | USA
sample row 3: University of California, Berkeley | USA```

- Table 1 (This table likely represents a list of scientific research projects, including the project ID, title of the project, and the lead scientist(s) responsible.):
```col: Project_ID | Project_Title | Lead_Scientist(s)
sample row 1: R009 | Soil regeneration systems | GeoBotanist
sample row 2: R002 | Quantum encryption | Prof. Alicia B
sample row 3: R006 | Renewable water filters | Ana N.```

- Table 2 (The table likely represents a list of researchers or experts, including their names, fields of expertise, and affiliations with educational institutions or research organizations.):
```col: ResearcherName | Expertise | Institution Affiliation
sample row 

=> plan: ```json
[
  {
    "Join Result": "Join_1",
    "Left Table": "Table_0",
    "Right Table": "Table_2",
    "Left Join Key": "University_Country",
    "Right Join Key": "Institution_Affiliation"
  },
  {
    "Join Result": "Join_2",
    "Left Table": "Join_1",
    "Right Table": "Table_1",
    "Left Join Key": "ResearcherName",
    "Right Join Key": "Lead_Scientist(s)"
  }
]
```


'```json\n[\n  {\n    "Join Result": "Join_1",\n    "Left Table": "Table_0",\n    "Right Table": "Table_2",\n    "Left Join Key": "University_Country",\n    "Right Join Key": "Institution_Affiliation"\n  },\n  {\n    "Join Result": "Join_2",\n    "Left Table": "Join_1",\n    "Right Table": "Table_1",\n    "Left Join Key": "ResearcherName",\n    "Right Join Key": "Lead_Scientist(s)"\n  }\n]\n```'

## Classify STD/semantic SQL

In [5]:
import json
operations_json = '\n[\n  {\n    "Join Result": "Join_1",\n    "Left Table": "Table_0",\n    "Right Table": "Table_2",\n    "Left Join Key": "University_Name",\n    "Right Join Key": "Institution_Affiliation"\n  },\n  {\n    "Join Result": "Join_2",\n    "Left Table": "Join_1",\n    "Right Table": "Table_1",\n    "Left Join Key": "ResearcherName",\n    "Right Join Key": "Lead_Scientist(s)"\n  }\n]\n'
operations = json.loads(operations_json)

In [6]:
classification_prompt = """You are a highly skilled data engineer. You are given:
- A description of a join operation between two tables.
- Sample values for each join key column from both tables.

Your task is to classify whether the join can be performed using a standard SQL join (e.g., matching IDs or exactly matching names), or if it requires a *semantic join*. A semantic join is needed when the values differ in representation — for example, abbreviations, name variations, different formats, or different languages — and require normalization, transformation, or external knowledge to align correctly.

Carefully examine the values. If they are *not exactly equal*, and some interpretation or resolution is needed to make the join work, it is a semantic join.

At the end of your reasoning, respond in the following format (for easy parsing):

- Operation classification: standard or semantic"""

In [46]:
prefix = "../../../data_src/synthetic_research_sem_joins"
suffix = "_enhanced.csv"
institutions = pd.read_csv(f"{prefix}/institutions{suffix}")
research_projects = pd.read_csv(f"{prefix}/research_projects{suffix}")
scientists = pd.read_csv(f"{prefix}/scientists{suffix}")

tables: dict[str, DataFrame] = {
    'Table_0': institutions,
    'Table_1': research_projects,
    'Table_2': scientists,
}

In [84]:
for op in operations:
    join_result: str = op['Join Result']
    left_table: str = op['Left Table']
    right_table: str = op['Right Table']
    left_join_key: str = op['Left Join Key']
    right_join_key: str = op['Right Join Key']

    left_key_samples = tables[left_table][left_join_key].sample(5, random_state=42)    
    right_key_samples = tables[right_table][right_join_key].sample(5, random_state=42)
    msg = [
        {'role': 'system', 'content': classification_prompt},
        {'role': 'user', 'content': f'- Samples of left join key ({left_join_key}): {left_key_samples}\n\n- Samples of right join key ({right_join_key}): {right_key_samples}'},
    ]
    classification_result = model.chat(msg)
    print(f"=> classification_result: {classification_result}")

=> classification_result: Operation classification: semantic

Reasoning: The values in the `University_Name` column from the left table and the `Institution_Affiliation` column from the right table do not match exactly. For example, "University of Illinois Urbana-Champaign" needs to be normalized to "UIUC", "University of California, Berkeley" needs to be normalized to "UC Berkeley", and "Georgia Georgia Institute of Technology" needs to be normalized to "Gatech". This requires some form of normalization or mapping to align the values correctly, indicating that this is a semantic join.


KeyError: 'Join_1'

## Actual Semantic Join

In [ ]:
# import json
# operations_json = '\n[\n  {\n    "Join Result": "Join_1",\n    "Left Table": "Table_0",\n    "Right Table": "Table_2",\n    "Left Join Key": "University_Name",\n    "Right Join Key": "Institution_Affiliation"\n  },\n  {\n    "Join Result": "Join_2",\n    "Left Table": "Join_1",\n    "Right Table": "Table_1",\n    "Left Join Key": "ResearcherName",\n    "Right Join Key": "Lead_Scientist(s)"\n  }\n]\n'
# operations = json.loads(operations_json)

In [1]:
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer('../llm/weight/bge-base', device='cuda')

In [3]:
import pandas as pd
prefix = "../../../data_src/synthetic_research_sem_joins"
suffix = "_enhanced.csv"
institutions = pd.read_csv(f"{prefix}/institutions{suffix}")
research_projects = pd.read_csv(f"{prefix}/research_projects{suffix}")
scientists = pd.read_csv(f"{prefix}/scientists{suffix}")

### Scientists - Institutions

In [127]:
alpha = 0.5
scores = dict()

In [128]:
left_table = scientists
right_table = institutions
uni_full_names = list(left_table['Institution_Affiliation'])
ENC_uni_full_names = embed_model.encode(uni_full_names)
uni_partial_names = list(right_table['University_Name'])
ENC_uni_partial_names = embed_model.encode(uni_partial_names)

In [130]:
from sentence_transformers import util
for i in range(len(uni_full_names)):
    print(f"=> Full name: {uni_full_names[i]}")
    for j in range(len(uni_partial_names)):
        print(f"==> Partial name: {uni_partial_names[j]}")
        cos_sim = util.cos_sim(ENC_uni_full_names[i], ENC_uni_partial_names[j])
        print(f"===> COSINE: {cos_sim}")
        if not scores.get(f"{uni_full_names[i]}-{uni_partial_names[j]}"):
            scores[f"{uni_full_names[i]}-{uni_partial_names[j]}"] = dict()
        scores[f"{uni_full_names[i]}-{uni_partial_names[j]}"]['cos'] = cos_sim[0].item()
    print("=" * 50)

=> Full name: MIT
==> Partial name: Massachusetts Institute of Technology
===> COSINE: tensor([[0.8395]])
==> Partial name: Stanford University
===> COSINE: tensor([[0.6451]])
==> Partial name: University of Chicago
===> COSINE: tensor([[0.6460]])
==> Partial name: California Institute of Technology
===> COSINE: tensor([[0.6845]])
==> Partial name: Cornell University
===> COSINE: tensor([[0.5861]])
==> Partial name: University of California, Berkeley
===> COSINE: tensor([[0.6292]])
==> Partial name: Harvard University
===> COSINE: tensor([[0.6516]])
==> Partial name: Georgia Institute of Technology
===> COSINE: tensor([[0.6672]])
==> Partial name: University of Illinois Urbana-Champaign
===> COSINE: tensor([[0.6274]])
==> Partial name: University of Washington
===> COSINE: tensor([[0.6175]])
=> Full name: Stanford
==> Partial name: Massachusetts Institute of Technology
===> COSINE: tensor([[0.5922]])
==> Partial name: Stanford University
===> COSINE: tensor([[0.9191]])
==> Partial name

In [110]:
from textdistance import damerau_levenshtein
for i in range(len(uni_full_names)):
    # print(f"=> Full name: {uni_full_names[i]}")
    for j in range(len(uni_partial_names)):
        # print(f"==> Partial name: {uni_partial_names[j]}")
        s1 = uni_full_names[i]
        s2 = uni_partial_names[j]
        dl = damerau_levenshtein(uni_full_names[i], uni_partial_names[j])
        metric = 1 - (dl / max(len(s1), len(s2)))
        scores[f"{s1}-{s2}"]['ed'] = metric
        # print(f"===> Metric: {metric} (DL: {dl})")

In [111]:
combined_scores = dict()
for i in scores.keys():
    cos_sim = scores[i]['cos']
    edit_dist = scores[i]['ed']
    combined_scores[i] = alpha * cos_sim + (1-alpha) * edit_dist

In [118]:
scores

{'MIT-Massachusetts Institute of Technology': {'cos': 0.8395031094551086,
  'ed': 0.08108108108108103},
 'MIT-Stanford University': {'cos': 0.6450727581977844, 'ed': 0.0},
 'MIT-University of Chicago': {'cos': 0.6459667086601257, 'ed': 0.0},
 'MIT-California Institute of Technology': {'cos': 0.6845030188560486,
  'ed': 0.05882352941176472},
 'MIT-Cornell University': {'cos': 0.5860555768013, 'ed': 0.0},
 'MIT-University of California, Berkeley': {'cos': 0.629231333732605,
  'ed': 0.0},
 'MIT-Harvard University': {'cos': 0.6516444683074951, 'ed': 0.0},
 'MIT-Georgia Institute of Technology': {'cos': 0.6672367453575134,
  'ed': 0.06451612903225812},
 'MIT-University of Illinois Urbana-Champaign': {'cos': 0.6274397373199463,
  'ed': 0.02564102564102566},
 'MIT-University of Washington': {'cos': 0.6175029277801514, 'ed': 0.0},
 'Stanford-Massachusetts Institute of Technology': {'cos': 0.5921640396118164,
  'ed': 0.10810810810810811},
 'Stanford-Stanford University': {'cos': 0.9191300868988

In [116]:
combined_scores

{'MIT-Massachusetts Institute of Technology': 0.46029209526809484,
 'MIT-Stanford University': 0.3225363790988922,
 'MIT-University of Chicago': 0.32298335433006287,
 'MIT-California Institute of Technology': 0.37166327413390665,
 'MIT-Cornell University': 0.29302778840065,
 'MIT-University of California, Berkeley': 0.3146156668663025,
 'MIT-Harvard University': 0.32582223415374756,
 'MIT-Georgia Institute of Technology': 0.3658764371948858,
 'MIT-University of Illinois Urbana-Champaign': 0.326540381480486,
 'MIT-University of Washington': 0.3087514638900757,
 'Stanford-Massachusetts Institute of Technology': 0.35013607385996226,
 'Stanford-Stanford University': 0.6700913592388755,
 'Stanford-University of Chicago': 0.3043314544927506,
 'Stanford-California Institute of Technology': 0.34904009454390583,
 'Stanford-Cornell University': 0.3276286456320021,
 'Stanford-University of California, Berkeley': 0.38463788172777963,
 'Stanford-Harvard University': 0.40855350097020465,
 'Stanford-

In [115]:
pair_keys = list(combined_scores.keys())
cos_sims = []
for key in pair_keys:
    cos_sims.append(combined_scores[key])
sorted_indices = sorted(range(len(cos_sims)), key=lambda i: cos_sims[i])
pair_keys_sorted = [pair_keys[i] for i in sorted_indices]
cos_sims_sorted = [cos_sims[i] for i in sorted_indices]
pair_keys_sorted[-10:]

['Harvard-Stanford University',
 'UIUC-University of Illinois Urbana-Champaign',
 'MIT-Massachusetts Institute of Technology',
 'UW-University of Washington',
 'Caltech-California Institute of Technology',
 'UChicago-University of Chicago',
 'UC Berkeley-University of California, Berkeley',
 'Cornell-Cornell University',
 'Harvard-Harvard University',
 'Stanford-Stanford University']

In [113]:
# LOTUS
# from collections import defaultdict

# def quantile_scores(sorted_similarities):
#     n = len(sorted_similarities)
#     score_to_indices = defaultdict(list)

#     # Step 1: Group indices of each unique similarity score
#     for idx, score in enumerate(sorted_similarities):
#         score_to_indices[score].append(idx)

#     # Step 2: Assign average ranks for ties and compute quantiles
#     quantiles = [0.0] * n
#     for score, indices in score_to_indices.items():
#         avg_rank = sum(indices) / len(indices)  # average rank for tied values
#         quantile = avg_rank / (n - 1) if n > 1 else 0.0  # avoid divide by zero
#         for idx in indices:
#             quantiles[idx] = quantile

#     return quantiles
# pair_keys = list(scores.keys())
# cos_sims = []
# for key in pair_keys:
#     cos_sims.append(scores[key]['cos'])
# sorted_indices = sorted(range(len(cos_sims)), key=lambda i: cos_sims[i])
# pair_keys_sorted = [pair_keys[i] for i in sorted_indices]
# cos_sims_sorted = [cos_sims[i] for i in sorted_indices]
# quantile_scores = quantile_scores(cos_sims_sorted)
# pair_keys_sorted[-10:]

### Researchers' Names

In [131]:
alpha = 0.5
scores = dict()

In [132]:
left_table = scientists
right_table = research_projects
uni_full_names = list(left_table['ResearcherName'])
ENC_uni_full_names = embed_model.encode(uni_full_names)
uni_partial_names = list(right_table['Lead_Scientist(s)'])
ENC_uni_partial_names = embed_model.encode(uni_partial_names)

In [133]:
from sentence_transformers import util
for i in range(len(uni_full_names)):
    print(f"=> Full name: {uni_full_names[i]}")
    for j in range(len(uni_partial_names)):
        print(f"==> Partial name: {uni_partial_names[j]}")
        cos_sim = util.cos_sim(ENC_uni_full_names[i], ENC_uni_partial_names[j])
        print(f"===> COSINE: {cos_sim}")
        if not scores.get(f"{uni_full_names[i]}-{uni_partial_names[j]}"):
            scores[f"{uni_full_names[i]}-{uni_partial_names[j]}"] = dict()
        scores[f"{uni_full_names[i]}-{uni_partial_names[j]}"]['cos'] = cos_sim[0].item()
    print("=" * 50)

=> Full name: Anna Newton
==> Partial name: Dr. A. Newton
===> COSINE: tensor([[0.6702]])
==> Partial name: Prof. Alicia B
===> COSINE: tensor([[0.4874]])
==> Partial name: A.B. Charles
===> COSINE: tensor([[0.4409]])
==> Partial name: C. Blackstone
===> COSINE: tensor([[0.4462]])
==> Partial name: Agritech Team Lead
===> COSINE: tensor([[0.4504]])
==> Partial name: Ana N.
===> COSINE: tensor([[0.5914]])
==> Partial name: E. J. Chem
===> COSINE: tensor([[0.4415]])
==> Partial name: SpaceX Collaborator
===> COSINE: tensor([[0.4474]])
==> Partial name: GeoBotanist
===> COSINE: tensor([[0.4716]])
==> Partial name: Underwater Tech Group
===> COSINE: tensor([[0.3817]])
=> Full name: Alicia Bell
==> Partial name: Dr. A. Newton
===> COSINE: tensor([[0.3975]])
==> Partial name: Prof. Alicia B
===> COSINE: tensor([[0.6512]])
==> Partial name: A.B. Charles
===> COSINE: tensor([[0.4146]])
==> Partial name: C. Blackstone
===> COSINE: tensor([[0.4045]])
==> Partial name: Agritech Team Lead
===> COS

In [134]:
from textdistance import damerau_levenshtein
for i in range(len(uni_full_names)):
    print(f"=> Full name: {uni_full_names[i]}")
    for j in range(len(uni_partial_names)):
        print(f"==> Partial name: {uni_partial_names[j]}")
        s1 = uni_full_names[i]
        s2 = uni_partial_names[j]
        dl = damerau_levenshtein(uni_full_names[i], uni_partial_names[j])
        metric = 1 - (dl / max(len(s1), len(s2)))
        scores[f"{s1}-{s2}"]['ed'] = metric
        print(f"===> Metric: {metric} (DL: {dl})")
    print("=" * 50)

=> Full name: Anna Newton
==> Partial name: Dr. A. Newton
===> Metric: 0.5384615384615384 (DL: 6)
==> Partial name: Prof. Alicia B
===> Metric: 0.0714285714285714 (DL: 13)
==> Partial name: A.B. Charles
===> Metric: 0.16666666666666663 (DL: 10)
==> Partial name: C. Blackstone
===> Metric: 0.23076923076923073 (DL: 10)
==> Partial name: Agritech Team Lead
===> Metric: 0.16666666666666663 (DL: 15)
==> Partial name: Ana N.
===> Metric: 0.4545454545454546 (DL: 6)
==> Partial name: E. J. Chem
===> Metric: 0.0 (DL: 11)
==> Partial name: SpaceX Collaborator
===> Metric: 0.1578947368421053 (DL: 16)
==> Partial name: GeoBotanist
===> Metric: 0.0 (DL: 11)
==> Partial name: Underwater Tech Group
===> Metric: 0.23809523809523814 (DL: 16)
=> Full name: Alicia Bell
==> Partial name: Dr. A. Newton
===> Metric: 0.15384615384615385 (DL: 11)
==> Partial name: Prof. Alicia B
===> Metric: 0.3571428571428571 (DL: 9)
==> Partial name: A.B. Charles
===> Metric: 0.16666666666666663 (DL: 10)
==> Partial name: C

In [135]:
combined_scores = dict()
for i in scores.keys():
    cos_sim = scores[i]['cos']
    edit_dist = scores[i]['ed']
    combined_scores[i] = alpha * cos_sim + (1-alpha) * edit_dist

In [136]:
scores

{'Anna Newton-Dr. A. Newton': {'cos': 0.6701552867889404,
  'ed': 0.5384615384615384},
 'Anna Newton-Prof. Alicia B': {'cos': 0.4873948097229004,
  'ed': 0.0714285714285714},
 'Anna Newton-A.B. Charles': {'cos': 0.4408598840236664,
  'ed': 0.16666666666666663},
 'Anna Newton-C. Blackstone': {'cos': 0.44616979360580444,
  'ed': 0.23076923076923073},
 'Anna Newton-Agritech Team Lead': {'cos': 0.45040541887283325,
  'ed': 0.16666666666666663},
 'Anna Newton-Ana N.': {'cos': 0.591449499130249, 'ed': 0.4545454545454546},
 'Anna Newton-E. J. Chem': {'cos': 0.4414910078048706, 'ed': 0.0},
 'Anna Newton-SpaceX Collaborator': {'cos': 0.447398841381073,
  'ed': 0.1578947368421053},
 'Anna Newton-GeoBotanist': {'cos': 0.4716251790523529, 'ed': 0.0},
 'Anna Newton-Underwater Tech Group': {'cos': 0.38174283504486084,
  'ed': 0.23809523809523814},
 'Alicia Bell-Dr. A. Newton': {'cos': 0.3974536955356598,
  'ed': 0.15384615384615385},
 'Alicia Bell-Prof. Alicia B': {'cos': 0.6512004733085632,
  'ed':

In [139]:
combined_scores

{'Anna Newton-Dr. A. Newton': 0.6043084126252394,
 'Anna Newton-Prof. Alicia B': 0.2794116905757359,
 'Anna Newton-A.B. Charles': 0.3037632753451665,
 'Anna Newton-C. Blackstone': 0.3384695121875176,
 'Anna Newton-Agritech Team Lead': 0.30853604276974994,
 'Anna Newton-Ana N.': 0.5229974768378518,
 'Anna Newton-E. J. Chem': 0.2207455039024353,
 'Anna Newton-SpaceX Collaborator': 0.30264678911158915,
 'Anna Newton-GeoBotanist': 0.23581258952617645,
 'Anna Newton-Underwater Tech Group': 0.3099190365700495,
 'Alicia Bell-Dr. A. Newton': 0.2756499246909068,
 'Alicia Bell-Prof. Alicia B': 0.5041716652257102,
 'Alicia Bell-A.B. Charles': 0.29063934584458667,
 'Alicia Bell-C. Blackstone': 0.24072762062916386,
 'Alicia Bell-Agritech Team Lead': 0.35697942475477856,
 'Alicia Bell-Ana N.': 0.37928530167449603,
 'Alicia Bell-E. J. Chem': 0.22704076631502673,
 'Alicia Bell-SpaceX Collaborator': 0.29940816681636007,
 'Alicia Bell-GeoBotanist': 0.19387798011302948,
 'Alicia Bell-Underwater Tech Grou

In [137]:
pair_keys = list(combined_scores.keys())
cos_sims = []
for key in pair_keys:
    cos_sims.append(combined_scores[key])
sorted_indices = sorted(range(len(cos_sims)), key=lambda i: cos_sims[i])
pair_keys_sorted = [pair_keys[i] for i in sorted_indices]
cos_sims_sorted = [cos_sims[i] for i in sorted_indices]
pair_keys_sorted[-10:]

['Charles B. Adams-A.B. Charles',
 'Gabriel Botani-GeoBotanist',
 'Alicia Bell-Prof. Alicia B',
 'S. X. Team-E. J. Chem',
 'Anna Newton-Ana N.',
 'Anna Newton-Dr. A. Newton',
 'Ana Newton-Dr. A. Newton',
 'Ana Newton-Ana N.',
 'Evan J. Chemistry-E. J. Chem',
 'Cynthia Blackstone-C. Blackstone']

In [138]:
# from collections import defaultdict

# def quantile_scores(sorted_similarities):
#     n = len(sorted_similarities)
#     score_to_indices = defaultdict(list)

#     # Step 1: Group indices of each unique similarity score
#     for idx, score in enumerate(sorted_similarities):
#         score_to_indices[score].append(idx)

#     # Step 2: Assign average ranks for ties and compute quantiles
#     quantiles = [0.0] * n
#     for score, indices in score_to_indices.items():
#         avg_rank = sum(indices) / len(indices)  # average rank for tied values
#         quantile = avg_rank / (n - 1) if n > 1 else 0.0  # avoid divide by zero
#         for idx in indices:
#             quantiles[idx] = quantile

#     return quantiles
# pair_keys = list(scores.keys())
# cos_sims = []
# for key in pair_keys:
#     cos_sims.append(scores[key]['cos'])
# sorted_indices = sorted(range(len(cos_sims)), key=lambda i: cos_sims[i])
# pair_keys_sorted = [pair_keys[i] for i in sorted_indices]
# cos_sims_sorted = [cos_sims[i] for i in sorted_indices]
# quantile_scores = quantile_scores(cos_sims_sorted)
# pair_keys_sorted[-10:]